# 06 · 实战一：MNIST 手写数字识别（全连接网络）

> 第一个完整项目。涵盖：数据加载 → 可视化 → 建模 → 训练 → 评估 → 可视化预测 → 保存。
> 建议完整跑一遍，理解每一步。

## 6.1 任务介绍

- **输入**：28×28 灰度手写数字图片
- **输出**：0–9 共 10 类
- **数据量**：训练 60000，测试 10000
- **评价**：准确率（好的模型 > 98%）

## 6.2 导入与配置

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 6.3 加载数据

用 `torchvision.datasets.MNIST`。transforms 做标准化（MNIST 全局 mean≈0.1307, std≈0.3081）。

In [ ]:
MEAN, STD = 0.1307, 0.3081

train_tf = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),   # 轻微增强
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
])

train_full = datasets.MNIST("./data", train=True, download=True, transform=train_tf)
test_set = datasets.MNIST("./data", train=False, download=True, transform=test_tf)

# 从训练集切 5000 做验证
val_size = 5000
train_set, val_set = random_split(
    train_full, [len(train_full) - val_size, val_size],
    generator=torch.Generator().manual_seed(42))

print("训练集:", len(train_set))
print("验证集:", len(val_set))
print("测试集:", len(test_set))

In [ ]:
BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

print("训练 batch 数:", len(train_loader))

# 查看一个 batch
images, labels = next(iter(train_loader))
print("images:", images.shape, images.dtype)
print("labels:", labels.shape, labels.dtype, "前 10 个:", labels[:10].tolist())

## 6.4 可视化数据

标准化后的图像要反标准化才能正常显示：`img * std + mean`。

In [ ]:
def denormalize(t, mean=MEAN, std=STD):
    return t * std + mean

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lab in zip(axes.ravel(), images[:16], labels[:16]):
    ax.imshow(denormalize(img.squeeze()), cmap="gray")
    ax.set_title(str(lab.item()))
    ax.axis("off")
plt.suptitle("MNIST 样本")
plt.tight_layout(); plt.show()

# 查看类别分布
all_labels = train_full.targets
counts = torch.bincount(all_labels)
print("各类别数量:", counts.tolist())

## 6.5 定义模型

一个带 BatchNorm 和 Dropout 的 MLP：

```
784 -> 512 -> BN -> ReLU -> Dropout -> 256 -> BN -> ReLU -> Dropout -> 10
```

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=(512, 256), num_classes=10, p_drop=0.3):
        super().__init__()
        layers = []
        in_dim = 28 * 28
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p_drop)]
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))     # 展平

model = MLP().to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\n参数量: {n_params:,}")

# 模拟一次前向
with torch.no_grad():
    out = model(torch.randn(4, 1, 28, 28).to(device))
print("输出形状:", out.shape)      # (4, 10)

> `flatten` 用的是 `x.view(N, -1)`，也可以直接用 `nn.Flatten()`。

## 6.6 定义训练组件

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, steps_per_epoch=len(train_loader), epochs=5)

## 6.7 训练循环

写成函数，每 epoch 记录 train/val 指标。

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n

In [ ]:
EPOCHS = 5
hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
    va_loss, va_acc = evaluate(model, val_loader, criterion)

    hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
    hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)

    if va_acc > best_acc:
        best_acc = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch}/{EPOCHS} | train_loss {tr_loss:.4f} acc {tr_acc:.4f} "
          f"| val_loss {va_loss:.4f} acc {va_acc:.4f} | {time.time()-t0:.1f}s")

print(f"\n最佳验证准确率: {best_acc:.4f}")
model.load_state_dict(best_state)

In [ ]:
# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["train_loss"], "o-", label="train")
axes[0].plot(hist["val_loss"], "s-", label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist["train_acc"], "o-", label="train")
axes[1].plot(hist["val_acc"], "s-", label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6.8 在测试集上评估

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"测试集 loss = {test_loss:.4f}  准确率 = {test_acc:.4f}")
print(f"错误样本数 ≈ {int((1-test_acc) * len(test_set))} / {len(test_set)}")

## 6.9 错误分析：看看模型错在哪

收集所有预测错误的样本，可视化。这是工程中非常重要的步骤。

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(device))
        all_logits.append(logits.cpu())
        all_labels.append(y)
    return torch.cat(all_logits), torch.cat(all_labels)

logits, labels = collect_predictions(model, test_loader)
probs = torch.softmax(logits, dim=1)
preds = logits.argmax(dim=1)
conf = probs.max(dim=1).values

wrong = (preds != labels).nonzero(as_tuple=True)[0]
print(f"错误样本数: {len(wrong)}")

fig, axes = plt.subplots(2, 8, figsize=(15, 4.5))
for ax, idx in zip(axes.ravel(), wrong[:16]):
    ax.imshow(denormalize(test_set[idx][0].squeeze()), cmap="gray")
    ax.set_title(f"真:{labels[idx].item()} 预:{preds[idx].item()}\n{conf[idx]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("预测错误样本")
plt.tight_layout(); plt.show()

## 6.10 混淆矩阵

看哪些数字容易被混淆（如 4↔9、3↔8）。

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(labels.numpy(), preds.numpy())

plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("预测"); plt.ylabel("真实"); plt.title("MNIST 混淆矩阵")
plt.show()

print(classification_report(labels.numpy(), preds.numpy(), digits=4))

## 6.11 置信度分析

- **高置信度但预测错** = 危险错误，模型很自信却错了。
- **低置信度** = 模型不确定，可以交给人工复核（实际系统常用）。

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(conf[preds == labels].numpy(), bins=50, alpha=0.7, label="正确", density=True)
plt.hist(conf[preds != labels].numpy(), bins=50, alpha=0.7, label="错误", density=True)
plt.xlabel("预测置信度"); plt.ylabel("密度"); plt.legend(); plt.title("置信度分布")
plt.show()

# 最危险的错误：置信度高但错
wrong_conf, wrong_idx = conf[wrong], wrong
order = wrong_conf.argsort(descending=True)
print("最自信的错误样本（前 5 个）:")
for k in order[:5]:
    i = wrong_idx[k]
    print(f"  真实 {labels[i].item()} | 预测 {preds[i].item()} | 置信度 {conf[i]:.4f}")

## 6.12 保存与加载模型

详见第 12 节，这里先用标准方式：

In [ ]:
torch.save(model.state_dict(), "mnist_mlp.pth")
print("已保存 mnist_mlp.pth")

# 加载
model2 = MLP().to(device)
model2.load_state_dict(torch.load("mnist_mlp.pth", map_location=device))
model2.eval()
_, acc2 = evaluate(model2, test_loader, criterion)
print("加载后测试准确率:", round(acc2, 4))

## 6.13 用模型做单张预测

模拟真实推理：传入一张图片，返回类别和概率。

In [ ]:
@torch.no_grad()
def predict(model, img_tensor):
    model.eval()
    if img_tensor.dim() == 3:
        img_tensor = img_tensor.unsqueeze(0)          # (1, C, H, W)
    img_tensor = img_tensor.to(device)
    logits = model(img_tensor)
    probs = torch.softmax(logits, dim=1)[0]
    pred = probs.argmax().item()
    return pred, probs.cpu()

# 随机挑几张测试图
idxs = [0, 1, 2, 3]
fig, axes = plt.subplots(1, len(idxs), figsize=(3 * len(idxs), 3))
for ax, i in zip(np.atleast_1d(axes), idxs):
    img, true_lab = test_set[i]
    pred, probs = predict(model, img)
    ax.imshow(denormalize(img.squeeze()), cmap="gray")
    ax.set_title(f"预测 {pred} (真 {true_lab})\np={probs[pred]:.3f}")
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 打印一张图的完整 10 类概率
img, _ = test_set[0]
pred, probs = predict(model, img)
for d in range(10):
    bar = "█" * int(probs[d] * 50)
    print(f"{d}: {probs[d]:.4f} {bar}")

## 6.14 练习

**练习 1**：把隐藏层改成 `(1024, 512, 256)`，参数量和准确率变化如何？

**练习 2**：去掉 BatchNorm，观察收敛速度。

**练习 3**：把 `Dropout(0.3)` 改成 `0.0` 和 `0.8`，比较验证准确率（体会欠/过拟合）。

**练习 4**：把优化器从 AdamW 换成 `SGD(lr=0.01, momentum=0.9)`，观察需要多少 epoch 才能到同样精度。

**练习 5**：统计每个数字的准确率（用混淆矩阵对角线 / 行和），哪个数字最难？

In [ ]:
# 练习 5：每个数字的准确率
per_class = cm.diagonal() / cm.sum(axis=1)
for d in range(10):
    print(f"数字 {d}: 准确率 {per_class[d]:.4f}  (样本 {cm.sum(axis=1)[d]})")
print("\n最难数字:", int(per_class.argmin()))

## 6.15 小结

完整流程回顾：

1. **数据**：下载、transform、划分 train/val/test、DataLoader。
2. **模型**：MLP（Flatten + Linear + BN + ReLU + Dropout）。
3. **训练**：AdamW + OneCycleLR + CrossEntropyLoss，监控 val。
4. **评估**：测试集准确率、混淆矩阵、置信度分析。
5. **保存**：`state_dict`；**推理**：`eval()` + `no_grad()`。
6. **错误分析**：找出最难/最易混淆的类别。

全连接网络在本任务上已能到 ~98%。下一节我们上 CNN，能到 99%+。

下一节：`07_卷积神经网络CNN.ipynb`